# Multiscale removal ablation on Kaggle

Follow-up to the BEMD primary ablation. That run's subtraction changed model inputs by less than 0.001, so its conditions trained on effectively identical images. This notebook removes finer, well-separated components at their true amplitude:

- **Gaussian octave bands 0-4** (difference of Gaussians, sigma 1, 2, 4, 8, 16 px)
- **FABEMD BIMFs 0-3** (fast and adaptive bidimensional EMD; median envelope windows 3, 7, 17, 33 px)

plus the `original` baseline: 10 conditions, same split, seed, epochs, model, loss, and the same three frame exclusions as the primary run.

Enable a **GPU** and **Internet**, attach only the raw MRI dataset (CMR-MULTI-CINE-4CH). No BEMD cache is needed: components are computed on the fly. Run the cells in order; only the training cell trains.

In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir(), "Run this notebook on Kaggle."

REPO = WORK / "cardiac-mri-4ch-segmentation"
COMMIT = "SET_AFTER_PUSH"  # commit containing src/preprocessing/multiscale.py
URL = "https://github.com/sancia-jw/cardiac-mri-4ch-segmentation.git"
assert COMMIT != "SET_AFTER_PUSH", "Pin COMMIT to the pushed commit first."

if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, f"Existing checkout is {head}; expected {COMMIT}."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
print("Repository and dependencies ready:", head)

Locate the raw MRI. Leave the override as None when one matching dataset is attached. DATA must contain 4CH_TR/image and 4CH_TR/anno.

The derived-image cache goes to /tmp so it is **not** saved as notebook output (the primary run saved ~41,000 cache files there, which made downloading results slow).

In [ ]:
RAW_OVERRIDE = None  # Example: "/kaggle/input/cmr-multi-cine-4ch"

matches = sorted({p.parent for p in INPUT.rglob("4CH_TR")
                  if (p / "image").is_dir() and (p / "anno").is_dir()})
if RAW_OVERRIDE:
    DATA = Path(RAW_OVERRIDE)
elif len(matches) == 1:
    DATA = matches[0]
else:
    raise RuntimeError(f"Expected one raw MRI root; found {[str(p) for p in matches]}. Set RAW_OVERRIDE.")

OUT = WORK / "multiscale_ablation"
LOGS = WORK / "multiscale_logs"
ENHANCED = Path("/tmp/multiscale_enhanced_cache")
CONFIG = REPO / "configs/multiscale_ablation.yaml"
EXCLUSIONS = REPO / "configs/bemd_exclusions.json"
SPLITS = REPO / "outputs/splits_4ch.csv"

assert (DATA / "4CH_TR/image").is_dir() and (DATA / "4CH_TR/anno").is_dir()
for path in (CONFIG, EXCLUSIONS, SPLITS):
    assert path.is_file(), f"Missing {path}"
for name, path in [("DATA", DATA), ("OUT", OUT), ("ENHANCED (not saved)", ENHANCED)]:
    print(f"{name}: {path}")

Check the GPU, conditions, controls, and exclusions. This does not construct or train a model.

`RUNS` defaults to all 10 conditions. If one session cannot fit them (Kaggle caps a session at 12 h), set `RUNS = PART_1` here, save a version, then run a second version with `RUNS = PART_2` and a different `OUT`. `original` is in both parts so each part has its own baseline.

In [ ]:
import json
import torch
import yaml

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)

cfg = yaml.safe_load(CONFIG.read_text())
ALL_RUNS = cfg["train"]["runs"]
assert cfg["train"]["catalog"] == "multiscale"
assert (cfg["train"]["epochs"], cfg["train"]["batch_size"], cfg["train"]["lr"], cfg["train"]["seed"]) == (15, 4, 0.001, 42)
assert len(ALL_RUNS) == 10 and ALL_RUNS[0] == "original"

exclusions = json.loads(EXCLUSIONS.read_text())
assert {(r["case_stem"], r["frame_idx"]) for r in exclusions["excluded_frames"]} == {
    ("CINE_4CH_023", 16), ("CINE_4CH_051", 65), ("CINE_4CH_085", 65),
}

PART_1 = ["original"] + [r for r in ALL_RUNS if "gband" in r]
PART_2 = ["original"] + [r for r in ALL_RUNS if "fabemd" in r]
RUNS = ALL_RUNS
print("Conditions this session:", RUNS)

Validate inputs with the actual CLI: case split, frame counts, and exclusions from raw headers. No training, evaluation, or decomposition.

In [ ]:
LOGS.mkdir(parents=True, exist_ok=True)

BASE = [
    sys.executable, "-u", str(REPO / "scripts/run_bemd_ablation.py"),
    "--config", str(CONFIG),
    "--data-root", str(DATA),
    "--exclusions", str(EXCLUSIONS),
    "--splits-csv", str(SPLITS),
    "--output-root", str(OUT),
    "--enhanced-cache-root", str(ENHANCED),
    "--device", "cuda",
    "--runs", *RUNS,
]

def run_logged(command, log_name):
    log_path = LOGS / log_name
    print("Running:", " ".join(command))
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f"Command exited with code {code}; see {log_path}")

VALIDATION_PASSED = False
run_logged(BASE + ["--validate-only"], "validation.log")
VALIDATION_PASSED = True
print("Validation passed. No model has been trained.")

**START TRAINING**: runs every condition in `RUNS` and its test evaluation. Refuses to overwrite existing run folders; use a fresh `OUT` for a new experiment.

In [ ]:
import time

assert VALIDATION_PASSED, "Run the validation cell successfully first."
existing = [OUT / r for r in RUNS if (OUT / r).exists()]
if existing:
    raise RuntimeError("Refusing to overwrite: " + ", ".join(str(p) for p in existing))

t0 = time.time()
run_logged(BASE, "training.log")
print(f"Completed {len(RUNS)} conditions in {(time.time() - t0) / 3600:.2f} h.")

Results. `delta_test_fg_vs_original` is test foreground Dice minus the baseline. With one seed, differences under ~0.005 are within run-to-run noise.

In [ ]:
import pandas as pd
from IPython.display import display, FileLink

summary_path = OUT / "ablation_summary.csv"
summary = pd.read_csv(summary_path)
assert set(summary["run_id"]) == set(RUNS)
display(summary[["run_id", "decomposition_method", "best_epoch", "val_dice_mean_foreground",
                 "test_dice_mean_foreground", "delta_test_fg_vs_original"]]
        .sort_values("test_dice_mean_foreground", ascending=False))

for name in ("ablation_summary.csv", "ablation_summary.json"):
    display(FileLink(str(OUT / name)))
display(FileLink(str(LOGS / "training.log")))
for run_id in RUNS:
    checkpoint = OUT / run_id / "checkpoints/best.pt"
    assert checkpoint.is_file(), f"Missing best checkpoint: {checkpoint}"
print("All best checkpoints present.")